# 02. Transformer Heads: Induction, OV/QK Circuits, MTP, and Registers

- **Category:** 05_architectures / 04_transformers_and_heads
- **Lineage & Citations:** Nelson Elhage et al., 2021 (*A Mathematical Framework for Transformer Circuits*); Catherine Olsson et al., 2022 (*Induction Heads*); Timothée Darcet et al., 2023 (*Vision Transformers Need Registers*); Fabian Gloeckle et al., 2024 (*Multi-Token Prediction*).
- **Purpose:** Decomposes black-box attention heads into interpretable bilinear circuits (QK for routing, OV for copying), and implements modern head enhancements including register scratchpads and parallel Multi-Token Prediction (MTP).
- **Topic Connections:** `🔗 Upstream Prerequisites:` [Attention Variants MHA MQA GQA MLA and DiffAttn](01_attention_variants_mha_mqa_gqa_mla_and_diffattn.ipynb) $\to$ `🔗 Downstream Syntheses:` [Flash Attention Online Softmax and Flex Attention](03_flash_attention_online_softmax_and_flex_attention.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> - **MTP on Low Data Scale:** Multi-token prediction requires large pretraining corpora (> 500B tokens); on small datasets, predicting multiple steps causes underfitting.
> - **Excessive Register Tokens:** Adding $> 32$ register tokens wastes sequence length context without improving feature map cleanliness.

## Mathematical Foundations & Governing Equations

### 1. Decoupled QK & OV Circuits
Attention decomposition into routing (QK) and copy (OV) linear maps:
$$
\mathbf{Scores} = (x_q W_Q) (x_k W_K)^\top \cdot \text{scale}, \quad A = \text{softmax}(\mathbf{Scores}), \quad \mathbf{Out} = A ((x_v W_V) W_O)
$$

### 2. Registers & Multi-Token Prediction (MTP)
Register token sequence augmentation and future token projection:
$$
X_{\text{aug}} = [R; X], \quad \hat{y}_{T+k} = x_T W_{\text{mtp}, k}^\top + b_k \quad (k \in 1..K)
$$

In [ ]:
import math
import torch
import torch.nn.functional as F
import safetensors.torch
from pathlib import Path
import core

env = core.setup_env(seed=42)
CKPT_REL = Path("weights/05_architectures/04_transformers_and_heads/02_transformer_heads_induction_ov_qk_circuits_mtp_and_registers.safetensors")
ckpt_path = next(p / CKPT_REL for p in (Path.cwd(), *Path.cwd().parents) if (p / CKPT_REL).exists())
weights = safetensors.torch.load_file(ckpt_path)

x_q, x_k, x_v = weights["x_q"], weights["x_k"], weights["x_v"]
scale = float(weights["scale"][0])
print(f"Loaded {len(weights)} golden tensors on {env.device}, tokens: {list(x_q.shape)}, scale: {scale:.4f}")

In [ ]:
def ov_qk_circuits(x_q, x_k, x_v, w_q, w_k, w_v, w_o, scale):
    q = x_q @ w_q
    k = x_k @ w_k
    v = x_v @ w_v
    ov = v @ w_o
    scores = torch.bmm(q, k.transpose(1, 2)) * scale
    attn_weights = F.softmax(scores, dim=-1)
    return torch.bmm(attn_weights, ov), attn_weights

def register_prepend_and_mtp(x_seq, registers, w_mtp):
    x_aug = torch.cat([registers, x_seq], dim=1)
    x_last = x_seq[:, -1, :]
    mtp_logits = torch.einsum("bd,kvd->bkv", x_last, w_mtp)
    return x_aug, mtp_logits

In [ ]:
# Quantitative Mathematical Assertions & Golden Parity Verification
circuit_out, circuit_attn = ov_qk_circuits(
    x_q, x_k, x_v, weights["w_q"], weights["w_k"], weights["w_v"], weights["w_o"], scale
)
err_o = float(torch.max(torch.abs(circuit_out - weights["circuit_out"])))
err_a = float(torch.max(torch.abs(circuit_attn - weights["circuit_attn"])))
assert err_o < 1e-6, f"Circuit out error: {err_o:.2e}"
assert err_a < 1e-6, f"Circuit attn error: {err_a:.2e}"

x_aug, mtp_logits = register_prepend_and_mtp(x_q, weights["registers"], weights["w_mtp"])
err_aug = float(torch.max(torch.abs(x_aug - weights["x_aug"])))
err_mtp = float(torch.max(torch.abs(mtp_logits - weights["mtp_logits"])))
assert err_aug < 1e-6, f"x_aug error: {err_aug:.2e}"
assert err_mtp < 1e-6, f"mtp_logits error: {err_mtp:.2e}"
print(f"Parity Verified: Circuits={err_o:.2e}, Attn={err_a:.2e}, MTP Logits={err_mtp:.2e}")

In [ ]:
import matplotlib.pyplot as plt
from core.benchmark import benchmark_fn, format_benchmark_table

b_circ = benchmark_fn(lambda: ov_qk_circuits(
    x_q, x_k, x_v, weights["w_q"], weights["w_k"], weights["w_v"], weights["w_o"], scale
), iters=50, name="OV/QK Circuits")
b_mtp = benchmark_fn(lambda: register_prepend_and_mtp(x_q, weights["registers"], weights["w_mtp"]), iters=50, name="MTP Heads")
print(format_benchmark_table([b_circ, b_mtp]))

# Visualization: QK Attention Map & MTP Prediction Probabilities
fig, ax = plt.subplots(1, 2, figsize=(8, 2.5), dpi=100)
ax[0].imshow(circuit_attn[0].detach().numpy(), cmap="viridis", aspect="auto")
ax[0].set_title("QK Circuit Attention Pattern")
ax[1].imshow(mtp_logits[0].detach().numpy(), cmap="magma", aspect="auto")
ax[1].set_title("MTP Future Step Logits [K, V]")
plt.tight_layout()
plt.show()

In [ ]:
# Pathological Misuse: Discarding Register Tokens During Generation
# Register tokens serve as attention sinks; stripping them at runtime breaks hidden state calibration!
registers_zero = torch.zeros_like(weights["registers"])
x_aug_zero, _ = register_prepend_and_mtp(x_q, registers_zero, weights["w_mtp"])
norm_diff = float(torch.norm(x_aug - x_aug_zero))
print(f"Norm difference when zeroing registers: {norm_diff:.2f}")
assert norm_diff > 0.1, "Zeroing register tokens must fundamentally alter input representations!"
print("Demonstrated requirement of preserving learned register embeddings across all inference steps.")

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Circuits Enable Mechanistic Interpretability:** Factoring attention into QK and OV separates token routing from feature movement.
2. **Registers Act as Attention Sinks:** Prepending learnable tokens prevents outlier artifact buildup in token embeddings.
3. **MTP Powers Speculative Acceleration:** Predicting future tokens in parallel provides free draft tokens for speculation.

---
### Cross-Domain Generalisation Matrix
- **Vision Segmentation (DINOv2):** Register tokens eliminate high-norm artifacts in image patch backbones.
- **Speculative Decoding Engines:** MTP heads propose multiple candidate tokens simultaneously to boost LLM serving speeds.
- **Mechanistic Safety Audits:** Tracking induction circuits reveals in-context learning versus rote memorization.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [02_transformer_heads_induction_ov_qk_circuits_mtp_and_registers.md](../../../markdown/05_architectures/04_transformers_and_heads/02_transformer_heads_induction_ov_qk_circuits_mtp_and_registers.md)
- **PyTorch 2.x Module:** [02_transformer_heads_induction_ov_qk_circuits_mtp_and_registers.py](../../../pytorch2/05_architectures/04_transformers_and_heads/02_transformer_heads_induction_ov_qk_circuits_mtp_and_registers.py)
- **Functional JAX Module:** [02_transformer_heads_induction_ov_qk_circuits_mtp_and_registers.py](../../../jax/05_architectures/04_transformers_and_heads/02_transformer_heads_induction_ov_qk_circuits_mtp_and_registers.py)
- **Candle Rust Module:** [02_transformer_heads_induction_ov_qk_circuits_mtp_and_registers.rs](../../../candle/05_architectures/04_transformers_and_heads/02_transformer_heads_induction_ov_qk_circuits_mtp_and_registers.rs)
- **Hardware Kernel:** [02_transformer_heads_induction_ov_qk_circuits_mtp_and_registers.rs](../../../kernels/05_architectures/04_transformers_and_heads/02_transformer_heads_induction_ov_qk_circuits_mtp_and_registers.rs)
- **Universal Golden SafeTensors:** [02_transformer_heads_induction_ov_qk_circuits_mtp_and_registers.safetensors](../../../weights/05_architectures/04_transformers_and_heads/02_transformer_heads_induction_ov_qk_circuits_mtp_and_registers.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous Topic: 01. Attention Variants MHA MQA GQA MLA and DiffAttn](01_attention_variants_mha_mqa_gqa_mla_and_diffattn.ipynb) | [Category Index](../../../README.md#05_architectures) | [Next Topic: 03. Flash Attention Online Softmax and Flex Attention](03_flash_attention_online_softmax_and_flex_attention.ipynb) &rarr;